# 01 — Análise e preparação do Dataset

Dataset fornecido: `dataset2_complete.csv`. Objetivo: classificação binária `human` vs `ai`.

**Regras do docente:** dataset em inglês; criação/seleção dos dados; divisão treino/validação/teste; documentação do pipeline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA=Path('../data/processed/dataset2_clean.csv')
df=pd.read_csv(DATA)
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nClasses:')
print(df['Label'].value_counts())
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicated texts:', df['Text'].duplicated().sum())

# Estatísticas
stats=df['Text'].str.findall(r"\b[\w'-]+\b").str.len()
print('\nWord count:')
print(stats.describe())

In [ ]:
df['word_count']=df['Text'].str.findall(r"\b[\w'-]+\b").str.len()
df['sentence_count']=df['Text'].str.count(r'[.!?]')
df.groupby('Label')[['word_count','sentence_count']].agg(['mean','std','min','max'])

In [ ]:
df['word_count'].plot(kind='hist',bins=12)
plt.title('Distribuição do número de palavras')
plt.xlabel('Palavras'); plt.ylabel('Frequência'); plt.show()

## Decisões metodológicas

- O ficheiro é separado por `;`, não por vírgula.
- Existe uma variação de rótulo `Ai`; ela é normalizada para `ai`.
- O dataset contém 100 exemplos, 51 `human` e 49 `ai` após normalização.
- Os textos têm entre 100 e 123 palavras, com média aproximada de 114 palavras, o que é muito próximo do intervalo externo de 100–120 palavras descrito no enunciado.
- Não há textos duplicados neste ficheiro.
- Por ser pequeno, o dataset é adequado para demonstrar o pipeline, mas é insuficiente para concluir que o modelo generaliza para produção. Para a submissão final, recomenda-se ampliar o treino com outras fontes autorizadas pelo docente.

In [ ]:
from src.models import stratified_split
train,val,test=stratified_split(df[['ID','Text','Label','label']], train=.70, val=.15, seed=42)
print(len(train), len(val), len(test))
print('Train:', train.Label.value_counts().to_dict())
print('Val:', val.Label.value_counts().to_dict())
print('Test:', test.Label.value_counts().to_dict())

train.to_csv('../data/processed/train.csv',index=False)
val.to_csv('../data/processed/validation.csv',index=False)
test.to_csv('../data/processed/test.csv',index=False)